In [ ]:
import kagglehub

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
%matplotlib inline

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()
target_column = 'Target'

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# drop missing vals


In [ ]:
# Task 2: Write your code here:
df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:
missing_values = df.isnull().sum()
print(missing_values)

In [ ]:
# Task 4: Write your code here:

from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["number"]).columns.drop(target_column)  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])



In [ ]:
# Task 5: Write your code here:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  df[target_column].hist()
  plt.show()

check_target_imbalance(df, target_column)

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)
%pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
f1_vals = []
acc_vals = []
n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled, because the data is imbalanced
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate the model
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)
  f1_vals.append(f1)
  acc_vals.append(accuracy)

print(f"F1 average score:  {np.mean(f1_vals):.4f}")
print(f"Accuracy average score:  {np.mean(acc_vals):.4f}")

In [ ]:
# Task 1: Write your code here:
imp = model.feature_importances_
abs_imp = np.abs(imp)
sorted_idx = np.argsort(abs_imp)
features = X.columns

plt.barh(features[sorted_idx], imp[sorted_idx])
plt.title('Cat Bost Feature Importances')
plt.xlabel('Importance Score')
plt.show()

In [ ]:
# Task 2: Write your code here:
g_features = []
for i in imp[sorted_idx]:
  if i > 10:
    g_features.append(i)
g_features

In [ ]:
# Task Bonus: Write your code here: